## Phase 0.3 – Demand model: data leakage and data quality

**1. Leakage through `order_count`.** The original model predicted the monthly quantity per category using `order_count`, the number of order lines in that category in the same month. This value is only known after the month has ended, and it is almost a direct measure of the target (correlation ≈ 0.86). It cannot be used for forecasting.

**2. Incomplete data after September 2017.** From October 2017 the number of rows and the ordered quantity drop sharply, and only a few categories remain. Moreover, only 17 of the 50 categories have orders in every month between January 2015 and September 2017. The honest demand model is therefore built on this window and on these 17 categories.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from src.data import load_data
from src.features import monthly_demand_panel

data = load_data()

In [2]:
# Rebuild the original demand dataset (same as in Supply_Chain_v2)
old = (data.groupby(["order_year", "order_month", "Category Name"])
           .agg(total_demand=("Order Item Quantity", "sum"),
                avg_price=("Product Price", "mean"),
                order_count=("Order Id", "count"))
           .reset_index())

X_old = pd.get_dummies(old.drop(columns="total_demand"), columns=["Category Name"])
y_old = old["total_demand"]
tr, te = train_test_split(np.arange(len(old)), test_size=0.2, random_state=42)

rows = []
for name, cols in [("with order_count", list(X_old.columns)),
                   ("without order_count", [c for c in X_old.columns if c != "order_count"])]:
    rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X_old.iloc[tr][cols], y_old.iloc[tr])
    mae = mean_absolute_error(y_old.iloc[te], rf.predict(X_old.iloc[te][cols]))
    rows.append({"setup": name, "MAE": round(mae, 1)})

print(pd.DataFrame(rows).to_string(index=False))
print("Correlation order_count vs demand:", round(old["order_count"].corr(old["total_demand"]), 3))

              setup  MAE
   with order_count 19.8
without order_count 61.1
Correlation order_count vs demand: 0.855


In [3]:
monthly = (data.assign(period=data["order_date"].dt.to_period("M"))
               .groupby("period")
               .agg(rows=("Order Id", "size"),
                    categories=("Category Name", "nunique"),
                    quantity=("Order Item Quantity", "sum")))
monthly.tail(8)

,rows,categories,quantity
period,,,
2017-06,4951,24,10194
2017-07,5318,24,11091
2017-08,5305,24,11095
2017-09,5189,26,10502
2017-10,2255,20,2490
2017-11,2055,8,2055
2017-12,2124,14,2124
2018-01,2123,10,2123


In [4]:
panel = monthly_demand_panel(data)
print("Categories:", panel["Category Name"].nunique(),
      "| Months:", panel["period"].nunique(),
      "| Rows:", len(panel))
panel.head()

Categories: 17 | Months: 33 | Rows: 561


,Category Name,period,demand
0,Baseball & Softball,2015-01,68
1,Baseball & Softball,2015-02,47
2,Baseball & Softball,2015-03,66
3,Baseball & Softball,2015-04,66
4,Baseball & Softball,2015-05,57
